In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

## Change Detection with TorchGeo Tutorial
_Written by: Harald Kristen_

In this tutorial, we demonstrate how to train a change detection model using TorchGeo. 

**What is Change Detection**: 
Change detection is the process of identifying differences between images of the same location captured at different times. It's a fundamental task in remote sensing with numerous real-world applications:

- **Urban Monitoring**: Track city growth, new construction, and infrastructure development
- **Disaster Response**: Assess damage from floods, earthquakes, wildfires, and hurricanes
- **Deforestation Tracking**: Monitor illegal logging and forest loss
- **Agriculture**: Detect crop changes, irrigation patterns, and land use shifts
- **Environmental Monitoring**: Track glacial retreat, coastal erosion, and wetland changes

In this tutorial, we'll train a binary change detection model on the [OSCD100](https://torchgeo.readthedocs.io/en/stable/api/datasets.html#oscd100) dataset to detect urban changes in bi-temporal Sentinel-2 imagery. Our model will learn to identify where new buildings have been constructed or existing ones removed.

It's recommended to run this notebook on Google Colab if you don't have your own GPU. Click the "Open in Colab" button above to get started.

## Setup

First, we install TorchGeo and TensorBoard.

In [ ]:
%pip install torchgeo tensorboard gdown

## Imports

Next, we import TorchGeo and any other libraries we need.

In [ ]:
%matplotlib inline
%load_ext tensorboard

import os
import tempfile

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from torchgeo.datamodules import OSCD100DataModule
from torchgeo.datasets import OSCD100
from torchgeo.trainers import ChangeDetectionTask

torch.set_float32_matmul_precision('medium')
L.seed_everything(0, workers=True)

## Visualize the Dataset

Let's download the dataset and look at some examples from OSCD100.

**About OSCD100**: This is a smaller, downsampled version of the full [OSCD](https://torchgeo.readthedocs.io/en/stable/api/datasets.html#oscd) (Onera Satellite Change Detection) dataset. It is designed for tutorials and quick experimentation. It contains 100 image pairs at 256×256 resolution with 60 training, 20 validation, and 20 test samples.

Each sample contains all 13 Sentinel-2 spectral bands, though we only use the RGB bands for simplicity in this notebook. For research and benchmarking, use the full OSCD dataset.

In [ ]:
import torchgeo.datamodules.oscd as oscd_dm

# OSCD100 is a downsampled version of OSCD, so its per-band statistics differ
# from the full dataset. We override the datamodule's defaults with values
# computed from the OSCD100 training split.
MEAN100 = {
    'B01': 1699.1854248046875,
    'B02': 1519.08349609375,
    'B03': 1481.2452392578125,
    'B04': 1542.990234375,
    'B05': 1708.63134765625,
    'B06': 2195.550048828125,
    'B07': 2444.430419921875,
    'B08': 2349.7373046875,
    'B8A': 2592.41845703125,
    'B09': 815.1251831054688,
    'B10': 16.794158935546875,
    'B11': 2331.2841796875,
    'B12': 1734.558837890625,
}
STD100 = {
    'B01': 361.23687744140625,
    'B02': 517.9739379882812,
    'B03': 668.9505004882812,
    'B04': 953.0056762695312,
    'B05': 899.5523071289062,
    'B06': 812.8031616210938,
    'B07': 859.7351684570312,
    'B08': 868.78369140625,
    'B8A': 898.2400512695312,
    'B09': 382.8373718261719,
    'B10': 8.891886711120605,
    'B11': 1158.5733642578125,
    'B12': 1069.1888427734375,
}

oscd_dm.MEAN = MEAN100
oscd_dm.STD = STD100

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'oscd100')
dataset = OSCD100(root=root, split='train', bands=OSCD100.rgb_bands, download=True)

sample = dataset[2]

fig = dataset.plot(sample, suptitle='OSCD100 Sample')
plt.show()

print(f'Dataset size: {len(dataset)} image pairs in the {dataset.split} split')
print(
    f'Image shape: {sample["image"].shape} (T, C, H, W) where T=2 timesteps, C=3 RGB bands'
)
print(f'Mask shape: {sample["mask"].shape}')

## Lightning Modules

TorchGeo uses [Lightning](https://lightning.ai/docs/pytorch/stable/) to organize the training code and setup the dataloader. The `OSCD100DataModule`:
1. Downloads the data
2. Sets up train, validation, and test dataloaders
3. Applies preprocessing and augmentation

The following variables can be modified to control training.

In [ ]:
batch_size = 8
num_workers = 4
max_epochs = 50
fast_dev_run = False

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'oscd100')
datamodule = OSCD100DataModule(
    root=root,
    bands=OSCD100.rgb_bands,  # RGB: B04 (Red), B03 (Green), B02 (Blue)
    batch_size=batch_size,
    num_workers=num_workers,
    download=True,
)

For training a *deep learning change detection model*, we use the `ChangeDetectionTask` class from `torchgeo.trainers`, which handles:
- Preprocessing and forwarding the bi-temporal image pair through the model
- Training with binary cross-entropy loss
- Computing metrics like accuracy, F1-score, and Jaccard Index


We use [**BTC** (Bitemporal Change Transformer)](https://torchgeo.readthedocs.io/en/stable/api/models/btc.html), a change-detection-specific architecture that uses a Swin Transformer backbone pretrained on Cityscapes followed by a UPerNet decoder. Unlike general segmentation models that simply concatenate both images into a single tensor, BTC processes them through separate encoder paths and fuses temporal features in a change-aware way.

Key configuration choices:
- **RGB bands** (`in_channels=3`): Red, Green, Blue for a fast tutorial training experiment
- **`weights=True`**: Load Cityscapes-pretrained Swin-Tiny encoder
- **`lr=0.0001`**
- **`loss='bce'`**: Binary cross-entropy loss

For other supported models and backbones, check the [trainers documentation](https://torchgeo.readthedocs.io/en/stable/api/trainers.html#torchgeo.trainers.ChangeDetectionTask).

In [ ]:
task = ChangeDetectionTask(
    model='btc',
    backbone='swin_tiny',
    weights=True,  # Cityscapes-pretrained Swin-Tiny encoder
    loss='bce',
    pos_weight=torch.tensor([10.0]),  # upweight the rare change class
    in_channels=3,  # RGB bands
    lr=0.0001,
)

## Training

Now we can train the model using Lightning's [Trainer](https://lightning.ai/docs/pytorch/stable/common/trainer.html), that allows us to easily do:

- Model checkpointing, that saves the best model based on a metric we define
- Early stopping to prevent overfitting
- TensorBoard logs metrics for visualization

In [ ]:
default_root_dir = os.path.join(tempfile.gettempdir(), 'experiments')
checkpoint_callback = ModelCheckpoint(
    monitor='val_AverageF1Score',
    mode='max',
    dirpath=default_root_dir,
    save_top_k=1,
    save_last=True,
)
early_stopping_callback = EarlyStopping(
    monitor='val_AverageF1Score', mode='max', min_delta=0.0, patience=10
)
logger = TensorBoardLogger(save_dir=default_root_dir, name='change_detection_logs')

In [ ]:
trainer = Trainer(
    callbacks=[checkpoint_callback, early_stopping_callback],
    log_every_n_steps=1,
    logger=logger,
    min_epochs=1,
    max_epochs=max_epochs,
)

**Note**: Training completes in ~90 seconds using ~ 2GB VRAM of your GPU.

In [ ]:
trainer.fit(model=task, datamodule=datamodule)

### Visualize Training with TensorBoard

Use TensorBoard to visualize metrics across epochs.

**In Google Colab**, run the cell below to launch TensorBoard inline.

**Locally**, run from your terminal `tensorboard --logdir /tmp/experiments` and navigate to http://localhost:6006

In [ ]:
%tensorboard --logdir "$default_root_dir"

## Evaluation

Now we evaluate the model on the test set.

### Understanding the Metrics

- **Accuracy**: Percentage of correctly classified pixels
- **F1 Score**: Harmonic mean of precision and recall
- **Jaccard Index (IoU)**: Intersection over Union between prediction and ground truth

In [ ]:
trainer.test(model=task, datamodule=datamodule, ckpt_path='best')

## Visualize Predictions

Let's visualize predictions on the test set using our best model. We run inference on the full 256×256 images rather than random crops, so the predictions align with the ground truth masks.

In [ ]:
# Use full 256x256 images — test_dataset applies RandomCrop so predictions
# from the dataloader wouldn't align with ground truth on direct access.
viz_dataset = OSCD100(root=root, split='test', bands=OSCD100.rgb_bands)

mean_t = datamodule.mean.view(1, 3, 1, 1)
std_t = datamodule.std.view(1, 3, 1, 1)

task.eval()


def show_sample(idx: int) -> None:
    sample = viz_dataset[idx]
    # Normalize manually: we call predict_step directly (no datamodule),
    # so the datamodule's augmentation pipeline does not run.
    x = (sample['image'].float() - mean_t) / std_t
    batch = {'image': x.unsqueeze(0).to(task.device)}
    with torch.no_grad():
        prob = task.predict_step(batch, 0).squeeze().cpu()
    pred_mask = (prob > 0.5).float()

    def to_rgb(img: torch.Tensor) -> np.ndarray:
        # viz_dataset loads only rgb_bands, so all channels are already RGB
        arr = img.float().numpy()
        lo, hi = np.percentile(arr, 2), np.percentile(arr, 98)
        return np.clip((arr - lo) / (hi - lo), 0, 1).transpose(1, 2, 0)

    _, axs = plt.subplots(1, 4, figsize=(20, 5))
    axs[0].imshow(to_rgb(sample['image'][0]))
    axs[0].set_title('Pre-change (T1)')
    axs[1].imshow(to_rgb(sample['image'][1]))
    axs[1].set_title('Post-change (T2)')
    axs[2].imshow(sample['mask'].squeeze(), cmap='gray', vmin=0, vmax=1)
    axs[2].set_title('Ground Truth')
    axs[3].imshow(pred_mask.squeeze(), cmap='gray', vmin=0, vmax=1)
    axs[3].set_title('Prediction')
    for ax in axs:
        ax.axis('off')
    plt.suptitle(f'Test Sample {idx}')
    plt.tight_layout()
    plt.show()


for idx in [1, 7, 13]:
    show_sample(idx)

### Interpreting the Results

With BTC (swin_tiny) you should expect:

- **Accuracy**: ~94–96%
- **F1 Score**: ~0.60–0.70
- **IoU (Jaccard Index)**: ~0.45–0.55

Change detection is hard due to class imbalance — most pixels show no change. F1 above 0.50 means the model is detecting real change rather than predicting everything as no-change.

Results may vary slightly between runs; Swin Transformer uses GPU operations that are non-deterministic even with a fixed seed.

### To Go Further

- Train on the full [OSCD](https://torchgeo.readthedocs.io/en/stable/api/datasets.html#oscd) dataset (24 city pairs) with full resolution
- Add all 13 Sentinel-2 bands (`in_channels=3` → `in_channels=13`, set `weights=False`)
- Train for 100+ epochs with a learning rate scheduler
- Try a larger backbone (`swin_small`, `swin_base`) for higher capacity